# Chapter 12: Safety Loading and Ruin

Code of Chapter 12 of *Practical Introduction to Non-Life Insurance Actuarial Mathematics* (F.-X. Cart-Tanneur). Run the cells from the top: later cells reuse the data and imports of earlier ones. Each comment after `#` gives the value printed in the book.

In [ ]:
# Setup: make the book's modules importable (works locally and in the browser)
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))

The walkthrough comes first, because it loads the data and imports used by the other examples of the chapter.

## 12.7 Python Walkthrough

In [ ]:
import numpy as np
from scipy import stats
from gidata import synthetic_portfolio
from gilossmodels import fit_mle
from giaggregate import discretize, panjer, grid_moments, grid_quantile
from giruin import (expected_value_premium, variance_premium, std_premium, exceedance,
                    adjustment_coefficient, exponential_R, exponential_ruin,
                    loading_for_ruin, capital_for_ruin, simulate_ruin)

# Section 12.1.3: premium principles on the book of 500 policies
_, claims = synthetic_portfolio()
L = fit_mle(claims["ultimate_amount"].to_numpy(), "lognormal")
X = stats.lognorm(L["sigma"], scale=np.exp(L["mu"]))      # scipy: s = sigma, scale = e^mu
g = panjer("poisson", discretize(X.cdf, 200, 2**13), lam=48.5)
m, v = grid_moments(g, 200)                                # 195919, 56213^2
P = std_premium(m, v, 0.1 * m / np.sqrt(v))                # 215511
exceedance(g, 200, P)                                      # 0.313

# Section 12.2.4: exponential claims, lambda = 50, mean 4,000, loading 10%
exponential_R(1 / 4000, 0.10)                              # 2.2727e-05
exponential_ruin(50_000, 1 / 4000, 0.10)                   # 0.2918
adjustment_coefficient(50, 220_000, lambda r: (1 / 4000) / (1 / 4000 - r), 1 / 4000)

# Section 12.3: simulation
simulate_ruin(50_000, 50, 220_000, lambda r, k: r.exponential(4000, k), 50, 20_000)  # 0.2921

# Section 12.4: the trade-off
loading_for_ruin(100_000, 1 / 4000, 0.01)                  # 0.2142
capital_for_ruin(1 / 4000, 0.10, 0.01)                     # 198434

## Exercise 1: Ruin with no capital

In [ ]:
from giruin import exponential_ruin, exponential_R
import numpy as np
b = 1 / 4000
exponential_ruin(0, b, 0.10), exponential_ruin(20_000, b, 0.10)   # 0.9091, 0.5770
np.log(2) / exponential_R(b, 0.10)                                 # 30498.48

## Exercise 2: The loading for a 5% ruin probability

In [ ]:
from giruin import loading_for_ruin
loading_for_ruin(100_000, 1 / 4000, 0.05)                          # 0.1299